# Assignment 2: Air Conditioner Controller using Fuzzy Logic (Mamdani)

In [ ]:
# !pip install scikit-fuzzy
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt

In [ ]:
# 1. Inputs and output with their ranges (universe of discourse)
temp     = ctrl.Antecedent(np.arange(10, 41, 1), 'temperature')   # degrees C
humidity = ctrl.Antecedent(np.arange(0, 101, 1), 'humidity')      # percent
power    = ctrl.Consequent(np.arange(0, 101, 1), 'power')         # AC power %

# 2. Membership functions (triangular)
temp['cold']        = fuzz.trimf(temp.universe, [10, 10, 22])
temp['comfortable'] = fuzz.trimf(temp.universe, [18, 24, 30])
temp['hot']         = fuzz.trimf(temp.universe, [26, 40, 40])

humidity['low']    = fuzz.trimf(humidity.universe, [0, 0, 50])
humidity['medium'] = fuzz.trimf(humidity.universe, [20, 50, 80])
humidity['high']   = fuzz.trimf(humidity.universe, [50, 100, 100])

power['low']    = fuzz.trimf(power.universe, [0, 0, 40])
power['medium'] = fuzz.trimf(power.universe, [30, 50, 70])
power['high']   = fuzz.trimf(power.universe, [60, 100, 100])

# 3. Rule base  (& = AND = min, | = OR = max)
rules = [
    ctrl.Rule(temp['cold'], power['low']),
    ctrl.Rule(temp['comfortable'] & humidity['low'], power['low']),
    ctrl.Rule(temp['comfortable'] & (humidity['medium'] | humidity['high']), power['medium']),
    ctrl.Rule(temp['hot'] & humidity['low'], power['medium']),
    ctrl.Rule(temp['hot'] & (humidity['medium'] | humidity['high']), power['high']),
]

# 4. Build the fuzzy inference system
ac = ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))

def get_power(t, h):
    ac.input['temperature'] = t
    ac.input['humidity'] = h
    ac.compute()   # fuzzification -> rules -> aggregation -> centroid defuzzification
    return ac.output['power']

In [ ]:
while True:
    print("\n========= AC CONTROLLER MENU =========")
    print("1. Compute AC power for given temperature and humidity")
    print("2. Run sample test cases")
    print("3. Show membership function graphs")
    print("4. Show rule base")
    print("0. Exit")
    ch = input("Enter your choice: ").strip()
    try:
        if ch == '1':
            t = float(input("Enter temperature (10-40 C): "))
            h = float(input("Enter humidity (0-100 %): "))
            if not (10 <= t <= 40 and 0 <= h <= 100):
                print("Values out of range")
                continue
            p = get_power(t, h)
            print(f"AC Power = {p:.2f} %")
            power.view(sim=ac); plt.show()
        elif ch == '2':
            for t, h in [(15, 30), (24, 60), (32, 70), (38, 90)]:
                print(f"Temp = {t} C, Humidity = {h} %  ->  AC Power = {get_power(t, h):.2f} %")
        elif ch == '3':
            temp.view(); humidity.view(); power.view(); plt.show()
        elif ch == '4':
            for i, r in enumerate(rules, 1):
                print(f"Rule {i}: {r}")
        elif ch == '0':
            print("Exiting...")
            break
        else:
            print("Invalid choice")
    except ValueError:
        print("Please enter numbers only")